<a href="https://colab.research.google.com/github/elhamod/BA305_Fall_2026/blob/main/Session%2007%20-%20Model%20Assessment%20II/lab3_model_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lab 3: Model Evaluation**

**Objective:**  predict whether patients have diabetes or not, based on bloodpressure, glucose, etc.

dataset: 'diabetes.csv'

In [ ]:
# Load the libraries used in this lab.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Splitting the data, cross-validation, and the model itself
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LogisticRegression

# Evaluation metrics
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef
from sklearn.metrics import roc_curve, roc_auc_score

In [ ]:
# Load the diabetes data and look at the first few patients.
df = pd.read_csv('https://raw.githubusercontent.com/AnalyticsArmory/data/main/diabetes.csv')
df.head()

In [ ]:
# Check the column types and confirm no values are missing.
df.info()

There are 768 rows and 9 columns. The first 8 columns represent the features (X) and the last column represent the target/label (y). 

In [ ]:
# Split the table into the 8 predictor columns (X) and the outcome we want to predict (y).
X = df.drop('Outcome', axis=1).values
y = df['Outcome'].values

## Train/Test Split

In [ ]:
# Hold out 30% of the patients as a test set the model never sees while training.
# random_state fixes the random draw so everyone in class gets the same split.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

In [ ]:
# Confirm how many rows landed in each half.
print('X_train rows =', len(X_train), 'X_test rows =', len(X_test))
print('y_train rows =', len(y_train), 'y_test rows =', len(y_test))

In [ ]:
# Compare the share of diabetic patients (Outcome = 1) in the full data and in each half.
print("share of 1's in full dataset =", round(sum(y) / len(y), 2))
print("share of 1's in y_train      =", round(sum(y_train) / len(y_train), 2))
print("share of 1's in y_test       =", round(sum(y_test) / len(y_test), 2))

### Stratifying the split

The three proportions above do not match. About 35% of all patients are diabetic, but the random draw left only 32% of them in the test set. That is bad luck, not a bug — and it distorts the test score, because the model is being graded on a test set that does not look like the population.

Passing `stratify=y` tells `train_test_split` to draw *within* each outcome group, so both halves keep the original share of positives and negatives.

In [ ]:
# Re-split the data, this time keeping the original share of positives in both halves.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=2, stratify=y)

In [ ]:
# The number of rows in each half is unchanged.
print('X_train rows =', len(X_train), 'X_test rows =', len(X_test))
print('y_train rows =', len(y_train), 'y_test rows =', len(y_test))

In [ ]:
# Check the proportions again -- all three should now match.
print("share of 1's in full dataset =", round(sum(y) / len(y), 2))
print("share of 1's in y_train      =", round(sum(y_train) / len(y_train), 2))
print("share of 1's in y_test       =", round(sum(y_test) / len(y_test), 2))

## Prediction model (skip details on how it works for this lecture)

### Probabilities vs. decisions

The model produces two different things, and the difference matters for the rest of this lab:

* `predict_proba` gives a **probability** — how likely this patient is to be diabetic.
* `predict` gives a **decision** — a 0 or a 1.

To turn a probability into a decision you need a **cutoff**, and `predict` silently uses 0.5. Nothing about this problem says 0.5 is the right choice; we come back to that in *Dealing with thresholds* below.

In [ ]:
# Fit a logistic regression model on the training half, then score the held-out test half.
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)

# Probability that each test patient is diabetic
y_probs = model.predict_proba(X_test)[:, 1]

# The model's 0/1 decision for each test patient
y_pred = model.predict(X_test)

In [ ]:
# Put each test patient's probability next to the 0/1 decision it produced.
y_df = pd.DataFrame({'probability': y_probs, 'prediction': y_pred})
y_df.head()

In [ ]:
# Plot each patient's probability against their true outcome to see how well the two groups separate.
plt.scatter(y_df['probability'], y_test)
plt.xlabel('Predicted probability of diabetes')
plt.ylabel('True outcome (0 or 1)')
plt.show()

## Confusion Matrix
Important: we are interested in prediction accuracy on new unseen data, therefore, the confusion matrix must compare the out-of-sample y outcomes from the testing array y_test, to the predicted outcomes y_pred which are obtained by using the X data from the X_test set.


`confusion_matrix` returns a bare 2×2 array with no labels, so it is easy to mix up which axis is which. Putting it into a labelled DataFrame makes the convention explicit: **rows are the true outcome, columns are what the model predicted.**

In [ ]:
# Build the confusion matrix and label the axes so rows and columns cannot be confused.
cm = confusion_matrix(y_test, y_pred)
pd.DataFrame(cm, index=['True 0', 'True 1'], columns=['Pred 0', 'Pred 1'])

Reading the table above:

* **True negative** (top-left) — healthy patients the model correctly cleared
* **False positive** (top-right) — healthy patients the model wrongly flagged as diabetic
* **False negative** (bottom-left) — diabetic patients the model missed
* **True positive** (bottom-right) — diabetic patients the model correctly caught


In [ ]:
# Show the same matrix as row percentages: what share of each true group the model got right.
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, normalize='true', cmap='Blues')
plt.title('Normalized confusion matrix')
plt.show()

**The standard classification metrics**

Every metric below is just a different ratio built from those same four counts:

* **Accuracy** is `(tp + tn) / total` — the share of all patients classified correctly.
* **Precision** is `tp / (tp + fp)` — of the patients we flagged as diabetic, how many really were. It measures the classifier's ability *not* to label a healthy patient as diabetic.
* **Recall** is `tp / (tp + fn)` — of the patients who really are diabetic, how many we caught. It measures the classifier's ability to find all the positive cases.
* **F1 score** is the harmonic mean of precision and recall, so it is only high when *both* are high.
* **MCC** (Matthews correlation coefficient) balances all four counts at once and stays honest when the two classes are very unequal in size.

In [ ]:
# Pull the four counts straight out of the confusion matrix and check them against the test-set totals.
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print('total patients      =', len(y_test))
print('actual positives    =', tp + fn, '| actual negatives    =', tn + fp)
print('predicted positives =', tp + fp, '| predicted negatives =', tn + fn)
print('true positives      =', tp, '| false positives     =', fp)
print('true negatives      =', tn, '| false negatives     =', fn)

## Metric functions from scikit-learn

Rather than computing each ratio by hand from the four counts above, `sklearn.metrics` provides a function for each one.

In [ ]:
# Compute the standard classification metrics at the model's default 0.5 cutoff.
accuracy  = accuracy_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)        # aka TPR, aka sensitivity
precision = precision_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)            # between 0 and 1
mcc       = matthews_corrcoef(y_test, y_pred)   # between -1 and 1

print('accuracy:', round(accuracy, 2),
      '| TPR:', round(recall, 2),
      '| precision:', round(precision, 2),
      '| f1:', round(f1, 2),
      '| MCC:', round(mcc, 2))

# **Dealing with thresholds**

Every metric above was computed at the default 0.5 cutoff. Move the cutoff and every one of those numbers changes: a lower cutoff flags more patients as diabetic, which catches more true cases (**recall goes up**) at the price of more false alarms (**precision goes down**).

So the cutoff is a business decision, not a statistical one.

In [ ]:
# Classify using a cutoff we choose ourselves instead of the default 0.5, and see what it costs.
my_threshold = 0.6
y_pred_custom = np.where(y_probs >= my_threshold, 1, 0)

print('at the 0.5 cutoff: recall =', round(recall_score(y_test, y_pred), 2),
      '| precision =', round(precision_score(y_test, y_pred), 2))
print('at the', my_threshold, 'cutoff: recall =', round(recall_score(y_test, y_pred_custom), 2),
      '| precision =', round(precision_score(y_test, y_pred_custom), 2))

**Experiment.** Re-run the cell above with `my_threshold` set to `0.3` instead of `0.6`. Which of recall and precision goes up, and which goes down? Explain why in terms of how many patients the model now flags as diabetic.

### Putting a value on each outcome

Picking a cutoff by feel is guesswork. Instead, attach a **value** to each of the four confusion-matrix cells and let the numbers choose. The matrix below uses the same layout as the confusion matrix (rows = true outcome, columns = prediction):

|  | Predicted healthy | Predicted diabetic |
| --- | --- | --- |
| **Truly healthy** | `+1` correctly cleared | `-1` false alarm, an unnecessary test |
| **Truly diabetic** | `-5` a missed diagnosis | `+2` caught and treated |

Note the asymmetry: missing a diabetic patient is set five times as costly as a false alarm. That single choice is what will drag the best cutoff well below 0.5.

In [ ]:
# Assign a dollar-like value to each cell of the confusion matrix, in the same layout: rows = true, columns = predicted.
val_matrix = np.array([[1, -1],
                       [-5, 2]])
val_matrix

In [ ]:
# Try every cutoff from 0 to 1 and record the total value each one would produce.
threshold_grid = np.linspace(0, 1, 101)
net_values = []

for thresh in threshold_grid:
    y_pred_at_thresh = np.where(y_probs >= thresh, 1, 0)
    cm_at_thresh = confusion_matrix(y_test, y_pred_at_thresh)
    net_values.append(np.sum(cm_at_thresh * val_matrix))

net_values = np.array(net_values)

In [ ]:
# Plot net value against the cutoff to see which cutoff pays off best.
plt.plot(threshold_grid, net_values)
plt.xlabel('Probability cutoff')
plt.ylabel('Net value')
plt.title('Net value at each cutoff')
plt.show()

In [ ]:
# Pick out the single cutoff with the highest net value.
best_idx = np.argmax(net_values)
best_threshold = threshold_grid[best_idx]

print('Max net value =', net_values[best_idx], '| Best probability cutoff =', best_threshold)

In [ ]:
# Redraw the scatter plot with the best cutoff marked, to see who now gets classified as diabetic.
plt.scatter(y_df['probability'], y_test)
plt.axvline(x=best_threshold, color='red', linestyle='--', label='best cutoff')
plt.xlabel('Predicted probability of diabetes')
plt.ylabel('True outcome (0 or 1)')
plt.legend()
plt.show()

**Question 1.** The best cutoff came out at 0.18, far below the default 0.5 — the red line sits well to the left, so patients with fairly low probabilities still get flagged as diabetic. Why did the value matrix push the cutoff down instead of up?

**Answer**

*Provide your answer here*


# **ROC (Receiver Operating Characteristic) curve**

The value matrix picked one cutoff for one specific set of costs. The **ROC curve** takes a step back and shows the whole trade-off at once: for every possible cutoff, what fraction of diabetic patients do we catch (true positive rate) and what fraction of healthy patients do we wrongly flag (false positive rate)?

A model that is only guessing traces the diagonal. The further the curve bulges toward the top-left corner, the better the model separates the two groups — whichever cutoff you eventually choose.

In [ ]:
# Compute the false and true positive rates at every possible cutoff, and draw the resulting curve.
roc_fpr, roc_tpr, roc_thresholds = roc_curve(y_test, y_probs)

plt.plot([0, 1], [0, 1], 'k--', label='random guessing')
plt.plot(roc_fpr, roc_tpr, label='logistic regression')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curve')
plt.legend()
plt.show()

In [ ]:
# Every point on the curve above comes from one cutoff -- here they are as a table.
# The first row's cutoff is above 1: that is sklearn's way of saying "predict everyone as 0".
pd.DataFrame({
    'cutoff': roc_thresholds,
    'tpr': roc_tpr.round(2),
    'fpr': roc_fpr.round(2),
})

In [ ]:
# Summarise the whole curve in one number: 1.0 is perfect, 0.5 is random guessing.
roc_auc_score(y_test, y_probs)

## Cross-validation

Everything so far rests on **one** train/test split. If that split happened to be lucky, the score is optimistic; if unlucky, pessimistic.

**K-fold cross-validation** removes that dependence. The data is cut into `k` equal folds; the model is trained on `k-1` of them and scored on the one left out, and this repeats until every fold has served as the test set once. The average of the `k` scores is a far more stable estimate, and the spread across folds tells you how sensitive the model is to which rows it happened to see.

* `shuffle=True` randomises the row order before cutting the folds.
* `random_state` fixes that shuffle so the result is reproducible.
* `scoring` accepts `'accuracy'`, `'f1'`, `'precision'`, and [many others](https://scikit-learn.org/stable/modules/model_evaluation.html).

In [ ]:
# Score the model on 5 different train/test splits instead of trusting a single one.
folds = KFold(n_splits=5, shuffle=True, random_state=0)
cv_scores = cross_val_score(model, X, y, scoring='accuracy', cv=folds)

print('accuracy on each fold:', cv_scores.round(3))
print('average accuracy:', round(np.mean(cv_scores), 3))

## Follow up Questions and Exercises

**Question 1.** The model's accuracy is 0.77, which sounds respectable. About 35% of patients in this dataset are diabetic. What accuracy would you get from a "model" that simply predicts *not diabetic* for every single patient — and what does that tell you about using accuracy to judge this model?

**Answer**

*Provide your answer here*


**Question 2.** The ROC curve and the AUC (0.82) never mention the value matrix, yet the value matrix is what picked our cutoff of 0.18. If AUC does not depend on the cutoff at all, what is it actually measuring, and when would you report it instead of a single metric like accuracy?

**Answer**

*Provide your answer here*


**Exercise 1.** The clinic changes its mind about the costs. Every patient it flags now goes for an expensive, invasive follow-up test, so false alarms have become costly — and because the whole population is re-screened every year, missing someone this time is no longer catastrophic. The new value matrix:

|  | Predicted healthy | Predicted diabetic |
| --- | --- | --- |
| **Truly healthy** | `+1` | `-3` a costly unnecessary test |
| **Truly diabetic** | `-2` | `+2` |

Re-run the threshold sweep with this matrix. Report the best cutoff, and explain in one sentence why it moved in the direction it did compared with the original 0.18.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*
